In [1]:
import csv, sqlite3, os
from sqlite3 import Error

In [2]:
os.remove('sc_election_2025.db')

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'sc_election_2025.db'

In [3]:
##Task 1
def db_create(db_file):
    if os.path.exists(db_file):
        conn = sqlite3.connect(db_file)
        print(f'{db_file} connected')
    else:
        conn = sqlite3.connect(db_file)
        print(f'{db_file} created and connected')

    c_sql = '''
                        CREATE TABLE `candidates` (
                            `candidate_id` TEXT,
                            `name` TEXT NOT NULL,
                            `cls` TEXT NOT NULL,
                            `gender` TEXT NOT NULL CHECK(gender IN ( 'male' , 'female' )),
                            PRIMARY KEY(`candidate_id`)
                        );'''
    gps_sql = '''
                    CREATE TABLE `groups` (
                        `group_id` TEXT,
                        `name` TEXT NOT NULL,
                        PRIMARY KEY(`group_id`)
                    );'''
    c_g_sql = '''
                CREATE TABLE `candidate_group` (
                    `candidate_id` TEXT,
                    `group_id` INTEGER,
                    PRIMARY KEY(`candidate_id`,`group_id`),
                    FOREIGN KEY(`candidate_id`) REFERENCES `candidates`(`candidate_id`),
                    FOREIGN KEY(`group_id`) REFERENCES `groups`(`group_id`)
                );'''
    votes_sql = '''
                CREATE TABLE `votes` (
                    `voter` TEXT,
                    `vote` INTEGER NOT NULL,
                    FOREIGN KEY(`vote`) REFERENCES `groups`(`group_id`),
                    PRIMARY KEY(`voter`)
                );'''
    
    try:
        conn.execute(c_sql)
        print('candidates table created')
    except Error as e:
        print(e)
    try:
        conn.execute(gps_sql)
        print('groups table created')
    except Error as e:
        print(e)
    try:
        conn.execute(c_g_sql)
        print('candidate_group table created')
    except Error as e:
        print(e)
    try:
        conn.execute(votes_sql)
        print('votes table created')
    except Error as e:
        print(e)
    conn.close()


In [4]:
db_create('sc_election_2025.db')

sc_election_2025.db created and connected
candidates table created
groups table created
candidate_group table created
votes table created


In [5]:
##Task 2
def import_data(csv_file):
    with open(csv_file, 'r') as csvfile:
        table = csv_file.split('.')[0]
        csvreader = csv.reader(csvfile)
        header = next(csvreader)
        v = ('?,'*(len(header)))[:-1]  
        sql = f'INSERT INTO {table} VALUES ({v})'
        print(sql)
        for row in csvreader:
            try:
                conn.execute(sql, row)
                conn.commit()
            except Error as e:
                print(row, e)

In [7]:
files = ['candidates.csv', 'groups.csv', 'candidate_group.csv', 'votes.csv']
conn = sqlite3.connect('sc_election_2025.db')
print(f'sc_election_2025.db connected')
for file in files:
    import_data(file)
conn.close()

sc_election_2025.db connected
INSERT INTO candidates VALUES (?,?,?,?)
['sc2025_01', 'Lee Loong Loong', '23J30', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_02', 'Prata Singh', '23J31', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_03', 'D Dave', '23J31', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_04', 'Sylvie Lim', '23J37', 'female'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_05', 'Ken Tiong', '23J33', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_06', 'Louisiana Chua', '23J32', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_07', 'Moon Sun', '23J34', 'female'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_08', 'She Ting Lu', '23J30', 'female'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_09', 'Jamil Punch', '23J33', 'male'] UNIQUE constraint failed: candidates.candidate_id
['sc2025_10', 'Leong Kar Wai', '23J35', 'male'] UNIQUE constrain

In [8]:
#Task 3
def get_details(conn, group_id):
    sql = '''
        SELECT groups.name, candidates.name FROM groups
            JOIN candidate_group ON groups.group_id = candidate_group.group_id
            JOIN candidates ON candidates.candidate_id = candidate_group.candidate_id
        WHERE groups.group_id = ?
    '''
    cur = conn.execute(sql, (group_id,))
    details = cur.fetchall()
    #print(details)
    print_out = f'{details[0][0]} - \n'
    for detail in details:
        print_out += detail[1] + '\n'
    print(print_out[:-1])
    return details

conn = sqlite3.connect('sc_election_2025.db')
get_details(conn, 1)
conn.close()

Student Voice - 
Lee Loong Loong
D Dave
Nnarnia Saladin


In [9]:
#Task 4
def voting_result(conn):
    sql = '''
        SELECT groups.name, groups.group_id, COUNT(*) FROM votes
            JOIN groups ON groups.group_id = votes.vote  
        GROUP BY votes.vote
        ORDER BY COUNT(*) DESC
    '''
    cur = conn.execute(sql)
    result = cur.fetchall()
    for row in result:
        print(f'{row[0]:20} ({row[1]}) - {row[2]}')
    return (result[0][1], result)

conn = sqlite3.connect('sc_election_2025.db')
print(voting_result(conn))
conn.close()

Student Voice        (1) - 69
Student Right        (2) - 62
Total Student        (3) - 56
RV Together          (5) - 55
RV Steady            (4) - 54
('1', [('Student Voice', '1', 69), ('Student Right', '2', 62), ('Total Student', '3', 56), ('RV Together', '5', 55), ('RV Steady', '4', 54)])


In [10]:
##Task 5
def find_missing(conn):
    sql = "SELECT voter FROM votes ORDER BY voter"
    cur = conn.execute(sql)
    voter_list = [ int(row[0]) for row in cur.fetchall() ]
    expected = 20250001
    end = 20250300
    missings = []
    
    for voter in voter_list:
        if voter == expected:
            expected += 1
        else:
            while expected != voter:
                missings.append(expected)
                expected += 1
            expected += 1
    if voter != end:
        while voter < end:
            voter += 1
            missings.append(voter)
    
    print(f'Total missing voters {len(missings)}')
    sn = 1
    for missing in missings:
        print(f'{sn:>2}: {missing}')
        sn += 1

        
conn = sqlite3.connect('sc_election_2025.db')
find_missing(conn)
conn.close()

Total missing voters 4
 1: 20250001
 2: 20250200
 3: 20250201
 4: 20250300
